# C-5 조건부 부품: 근거 장부 상세화 · 동적 압축량 (개발 세트)

담당 C (조환호). 같은 라운드 `r8`에서 동결 M(`m`), M + 근거 장부(`m_ledger`), M + 동적 압축량(`m_dynamic_m`)을 개발 세트 60문항씩 차례로 돌립니다. 채택 기준은 `docs/experiments/2026-11-26-c-conditional.md`에 실행 전에 적어 두었습니다.

**시작 전:** `런타임 → 런타임 유형 변경 → T4 GPU`를 고르고 저장하세요. 셀을 위에서부터 하나씩 실행합니다.
세션이 끊기면(런타임이 초기화되지 않았다면) 2·4번 셀을 다시 실행한 뒤 6번 셀을 다시 실행하면 이어서 돕니다.

## 1. GPU 확인 (T4가 보여야 합니다)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. 코드 받기 (`HwanHo` 최신)

In [ ]:
import os
REPO = "/content/Intelligence-Software-Project"
if not os.path.exists(REPO):
    !git clone https://github.com/minyi-k03/Intelligence-Software-Project.git {REPO}
%cd {REPO}
!git fetch -q origin
!git checkout -q HwanHo && git pull -q --ff-only origin HwanHo && git log --oneline -1
assert os.path.isfile("src/meba/conditional/ledger.py"), "C-5 코드가 없다. HwanHo가 push됐는지 확인할 것"

## 3. 의존성 설치

In [ ]:
!pip install -q -r requirements.txt

## 4. Ollama 설치·실행, 모델 받기·미리 불러 두기 (모델 태그는 `configs/default.yaml`에서 읽음)

In [ ]:
!apt-get -qq install -y zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh
import subprocess, time, yaml
subprocess.Popen(["ollama", "serve"], stdout=open("/content/ollama.log", "w"), stderr=subprocess.STDOUT)
time.sleep(5)
MODEL_TAG = yaml.safe_load(open("configs/default.yaml", encoding="utf-8"))["model"]["tag"]
print("모델:", MODEL_TAG)
!ollama pull {MODEL_TAG}
# 첫 문항이 모델 로딩 때문에 120초 제한에 걸리지 않도록 미리 메모리에 올린다 (측정 밖 호출).
!ollama run {MODEL_TAG} "Reply with OK." 
!ollama ps

## 5. 동작 확인 (각 2문항, git에서 제외되는 임시 폴더)

In [ ]:
!python -m pytest tests/test_conditional.py tests/test_judge.py -q 2>&1 | tail -1
!python -m scripts.run_eval --verify 2 --pipelines m,m_ledger,m_dynamic_m

## 6. 본 실행 (`r8`, 60문항 × 3). 끊기면 이 셀을 다시 실행

In [ ]:
!python -m scripts.run_eval --bundle dev --round r8 --pipelines m,m_ledger,m_dynamic_m

## 7. 빠른 요약

In [ ]:
import json
from pathlib import Path
for name in ("m", "m_ledger", "m_dynamic_m"):
    rows = [json.loads(l) for l in Path(f"results/runs/dev/r8/{name}.jsonl").open(encoding="utf-8")]
    cost = [r for r in rows if not r["is_warmup"]]
    by = lambda t: sum(r["correct"] for r in rows if (r["type"] + ("-" + r["subtype"] if r["type"] == "squad2" else "")) == t)
    hp = [r for r in cost if r["type"] == "hotpotqa"]
    print(f"{name:12} 정답 {sum(r['correct'] for r in rows)}/60 | GSM8K {by('gsm8k')} HotpotQA {by('hotpotqa')} SQuAD답 {by('squad2-answerable')} 답없음 {by('squad2-unanswerable')}"
          f" | 호출 {sum(r['n_calls'] for r in cost)/len(cost):.2f} | HotpotQA 호출 {sum(r['n_calls'] for r in hp)/len(hp):.2f}"
          f" | 입력 토큰 {sum(sum(c['input_tokens'] for c in r['calls']) for r in cost)/len(cost):.0f} | 오류·시간초과 {sum((r['error'] is not None) or r['timeout'] for r in rows)}")

## 8. 결과 내려받기

내려받은 zip은 다운로드 폴더에 그대로 두시면 됩니다.

In [ ]:
import shutil
from google.colab import files
shutil.make_archive("/content/c5_r8", "zip", "results/runs/dev/r8")
files.download("/content/c5_r8.zip")